In [1]:
import cv2
import os
from tqdm import tqdm
import shutil
import matplotlib.pyplot as plt

In [2]:
ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"

In [3]:
def save_heatmap(img, path, cmap='grey'):
    plt.imsave(path, img, cmap=cmap)

In [4]:
import json
import numpy as np
import cv2
import skimage.measure as skms


def get_regions_from_mask(mask, min_area=0):
    """Extract connected regions from a binary mask."""
    label = skms.label(mask)
    props = skms.regionprops(label)
    
    regions = []
    for i, prop in enumerate(props):
        if prop.area >= min_area:
            region_mask = (label == i + 1).astype(np.uint8)
            regions.append({
                'mask': region_mask,
                'bbox': prop.bbox,
                'area': prop.area
            })
    return regions

In [5]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def calculate_iou(region_mask, bbox, bbox_format="coco", epsilon=1e-8, debug=False):
    """
    Compute IoU between a region mask and a bounding box.
    
    Args:
        region_mask (np.ndarray): Binary mask of region (H x W).
        bbox (list/tuple): Bounding box coordinates.
            - COCO format: [x, y, w, h]
            - YOLO format: [x_center, y_center, w, h] (normalized to [0,1])
        bbox_format (str): "coco" or "yolo".
        epsilon (float): Small constant to avoid division by zero.
        debug (bool): If True, visualize the region mask, bbox, and overlaps.
    """
    H, W = region_mask.shape
    
    if bbox_format == "coco":
        x1, y1, w, h = bbox
        x1, y1, x2, y2 = int(x1), int(y1), int(x1 + w), int(y1 + h)

    elif bbox_format == "yolo":
        # YOLO format is normalized: [x_center, y_center, w, h]
        x_center, y_center, w, h = bbox
        x_center, y_center, w, h = x_center * W, y_center * H, w * W, h * H
        x1 = int(x_center - w / 2)
        y1 = int(y_center - h / 2)
        x2 = int(x_center + w / 2)
        y2 = int(y_center + h / 2)

    else:
        raise ValueError("bbox_format must be either 'coco' or 'yolo'")

    # Clip to image boundaries
    if bbox_format == 'yolo':
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)

    bbox_mask = np.zeros_like(region_mask, dtype=np.uint8)
    bbox_mask[y1:y2, x1:x2] = 1

    intersection = np.logical_and(region_mask, bbox_mask).sum()
    union = np.logical_or(region_mask, bbox_mask).sum()
    iou = intersection / (union + epsilon)

    if debug:
        # Prepare visualization
        vis = np.zeros((H, W, 3), dtype=np.uint8)

        # Region mask in red
        vis[region_mask.astype(bool)] = [255, 0, 0]

        # Bbox mask in green
        vis[bbox_mask.astype(bool)] = [0, 255, 0]

        # Intersection in yellow
        intersection_mask = np.logical_and(region_mask, bbox_mask)
        vis[intersection_mask] = [255, 255, 0]

        # Draw bbox rectangle outline
        vis = cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 255, 255), 1)

        plt.figure(figsize=(6,6))
        plt.imshow(vis)
        plt.title(f"IoU = {iou:.4f}")
        plt.axis("off")
        plt.show()

    return iou


In [6]:
def match_region(region_mask, anns, threshold=0.5, mode='coco'):
    """Check if region matches any annotation (IoU ≥ threshold)."""
    for ann in anns:
        iou = calculate_iou(region_mask, ann['bbox'], bbox_format=mode, debug=False)
        # print(iou)
        if iou >= threshold:
            return True
    return False



In [7]:
import os

def tally_tp_yolo(label_dir, image_filename, category_ids, mask, threshold=0.5):
    regions = get_regions_from_mask(mask)
    label_path = os.path.join(label_dir, image_filename + '.txt')
    
    anns = []
    with open(label_path, "r") as f:
        i = 0
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            if class_id in category_ids:
                x_center, y_center, w, h = map(float, parts[1:5])
                anns.append({
                    'id': i,
                    'bbox': [x_center, y_center, w, h],
                    'category_id': class_id,
                    'area': w * h
                })
                i += 1

    # If there are no ground-truth anns, return zeros (no TP / FN)
    if not anns:
        return 0, 0

    # track which ground-truth anns have been matched (True => TP)
    ann_intersected = {ann['id']: False for ann in anns}

    # small epsilon to detect any overlap (IoU > 0)
    overlap_eps = 1e-6

    fp = 0
    
    for r in regions:
        # collect all bboxes that have any overlap with this region
        intersecting_anns = []
        for ann in anns:
            # use a tiny threshold to detect any overlap
            if match_region(r['mask'], [ann], threshold=overlap_eps, mode='yolo'):
                intersecting_anns.append(ann)

        if not intersecting_anns:
            # nothing to do for this region (no GT bbox touches it)
            fp += 1
            continue

        # single intersecting bbox: test with configured threshold
        if len(intersecting_anns) == 1:
            ann = intersecting_anns[0]
            if match_region(r['mask'], [ann], threshold=threshold, mode='yolo'):
                ann_intersected[ann['id']] = True
            else:
                fp += 1 # Mask region does not satisfy threshold (is a false positive)
        else:
            # multiple intersecting bboxes -> merge and test the merged bbox
            merged_ann = merge_yolo_bboxes(intersecting_anns)
            if match_region(r['mask'], [merged_ann], threshold=threshold, mode='yolo'):
                # mark all intersecting anns as detected
                for ann in intersecting_anns:
                    ann_intersected[ann['id']] = True
            else:
                fp += 1 # Mask region does not satisfy threshold (is a false positive)

    tp = sum(1 for v in ann_intersected.values() if v)
    fn = sum(1 for v in ann_intersected.values() if not v)
    return tp, fn, fp


def merge_yolo_bboxes(anns):
    """
    Merge multiple YOLO-format bboxes into a single bbox.
    Each ann['bbox'] is [xc, yc, w, h] (same format you read from labels).
    Returns an ann dict compatible with match_region (same keys as input anns).
    """
    xs = []
    ys = []
    for ann in anns:
        xc, yc, w, h = ann['bbox']
        x1 = xc - w / 2.0
        y1 = yc - h / 2.0
        x2 = xc + w / 2.0
        y2 = yc + h / 2.0
        xs.extend([x1, x2])
        ys.extend([y1, y2])

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    merged_w = x2 - x1
    merged_h = y2 - y1
    merged_xc = (x1 + x2) / 2.0
    merged_yc = (y1 + y2) / 2.0

    return {
        'bbox': [merged_xc, merged_yc, merged_w, merged_h],
        'category_id': anns[0]['category_id'],
        'area': merged_w * merged_h
    }


In [17]:
d_types = ['Test']

patch_types = [
    ['Naturalistic1', 'Naturalistic2', 'Naturalistic3',
    'Naturalistic4', 'Naturalistic5', 'Naturalistic6'],
    ['TSEA1', 'TSEA2']
]

test_types = ['Naturalistic5', 'Naturalistic6', 'TSEA2']

savefig_path = os.path.join(ROOT_DIR, 'results_cd_grey_test_final_eval_ablation')

for d_type in d_types:
    for t in patch_types:
        for patch in t:
            s1_tp = 0
            s1_fn = 0
            s2_tp = 0
            s2_fn = 0
            s3_tp = 0
            s3_fn = 0
            s4_tp = 0
            s4_fn = 0
            if d_type == 'Train' and patch in test_types:
                continue
            if d_type == 'Train':
                label_dir = 'labels'
            if d_type == 'Test':
                label_dir = 'labels_w_adv'
    
            eval_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, 'images')
            label_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, label_dir)
            heatmap_path = os.path.join(ROOT_DIR, 'results_cd_grey_test_final_eval')
            
            for fname in tqdm(os.listdir(eval_path), desc="Evaluating images"):
                if not fname.endswith('.jpg'):
                    continue

                
                path = os.path.join(heatmap_path, f'{fname.replace(".jpg", ".jpg")}_cd.png')
                no_morph = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.uint8)
                path = os.path.join(heatmap_path, f'{fname.replace(".jpg", ".jpg")}_cd_o.png')
                op = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.uint8)
                path = os.path.join(heatmap_path, f'{fname.replace(".jpg", ".jpg")}_cd_o_c.png')
                op_cl = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.uint8)
                path = os.path.join(heatmap_path, f'{fname.replace(".jpg", ".jpg")}_cd_o_c_o.png')
                op_cl_op = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.uint8)

                _, no_morph_thresh = cv2.threshold(
                    no_morph, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )
                _, op_thresh = cv2.threshold(
                    op, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )
                _, op_cl_thresh = cv2.threshold(
                    op_cl, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )
                _, op_cl_op_thresh = cv2.threshold(
                    no_morph, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )
                
                save_heatmap(no_morph_thresh, os.path.join(savefig_path, fname + "_cd_ablate.png"))
                save_heatmap(op_thresh, os.path.join(savefig_path, fname + "_cd_o_ablate.png"))
                save_heatmap(op_cl_thresh, os.path.join(savefig_path, fname + "_cd_o_c_ablate.png"))
                save_heatmap(op_cl_op_thresh, os.path.join(savefig_path, fname + "_cd_o_c_o_ablate.png"))
                

                no_morph_t_tp, no_morph_t_fn, _ = tally_tp_yolo(
                    label_dir=label_path,
                    image_filename=os.path.splitext(fname)[0],
                    category_ids=[1],
                    mask=no_morph_thresh,
                    threshold=0.5
                )
                s1_tp += no_morph_t_tp
                s1_fn += no_morph_t_fn
                
                op_t_tp, op_t_fn, _ = tally_tp_yolo(
                    label_dir=label_path,
                    image_filename=os.path.splitext(fname)[0],
                    category_ids=[1],
                    mask=op_thresh,
                    threshold=0.5
                )
                s2_tp += op_t_tp
                s2_fn += op_t_fn
                
                op_cl_t_tp, op_cl_t_fn, _ = tally_tp_yolo(
                    label_dir=label_path,
                    image_filename=os.path.splitext(fname)[0],
                    category_ids=[1],
                    mask=op_cl_thresh,
                    threshold=0.5
                )
                s3_tp += op_cl_t_tp
                s3_fn += op_cl_t_fn
                
                op_cl_op_t_tp, op_cl_op_t_fn, _ = tally_tp_yolo(
                    label_dir=label_path,
                    image_filename=os.path.splitext(fname)[0],
                    category_ids=[1],
                    mask=op_cl_op_thresh,
                    threshold=0.5
                )
                s4_tp += op_cl_op_t_tp
                s4_fn += op_cl_op_t_fn
                
            s1_recall = (s1_tp / (s1_tp + s1_fn)) * 100.0
            s2_recall = (s2_tp / (s2_tp + s2_fn)) * 100.0
            s3_recall = (s3_tp / (s3_tp + s3_fn)) * 100.0
            s4_recall = (s4_tp / (s4_tp + s4_fn)) * 100.0
            print(f"Recall of scenario 1: {s1_recall}\nRecall of scenario 2: {s2_recall}\nRecall of scenario 3: {s3_recall}\nRecall of scenario 4: {s4_recall}\n")
                

Evaluating images: 100%|█████████████████████████████████████████████████████████| 50/50 [04:37<00:00,  5.56s/it]


Recall of scenario 1: 82.53968253968253
Recall of scenario 2: 80.95238095238095
Recall of scenario 3: 73.01587301587301
Recall of scenario 4: 82.53968253968253



Evaluating images: 100%|█████████████████████████████████████████████████████████| 50/50 [04:54<00:00,  5.89s/it]


Recall of scenario 1: 93.24324324324324
Recall of scenario 2: 97.2972972972973
Recall of scenario 3: 90.54054054054053
Recall of scenario 4: 93.24324324324324



Evaluating images: 100%|█████████████████████████████████████████████████████████| 50/50 [05:29<00:00,  6.59s/it]


...

Recall of scenario 1: 84.0
Recall of scenario 2: 86.0
Recall of scenario 3: 80.0
Recall of scenario 4: 84.0



Evaluating images: 100%|█████████████████████████████████████████████████████████| 50/50 [04:27<00:00,  5.35s/it]


Recall of scenario 1: 93.54838709677419
Recall of scenario 2: 96.7741935483871
Recall of scenario 3: 93.54838709677419
Recall of scenario 4: 93.54838709677419



Evaluating images: 100%|█████████████████████████████████████████████████████████| 80/80 [06:18<00:00,  4.73s/it]

Recall of scenario 1: 92.66055045871559
Recall of scenario 2: 93.57798165137615
Recall of scenario 3: 88.9908256880734
Recall of scenario 4: 92.66055045871559

